# Transfer learning with Hoplite-based embedding storage

## Run this tutorial

This tutorial is more than a reference! It's a Jupyter Notebook which you can run and modify on Google Colab or your own computer.

|Link to tutorial|How to run tutorial|
| :- | :- |
| [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kitzeslab/opensoundscape/blob/master/docs/tutorials/train_cnn.ipynb) | The link opens the tutorial in Google Colab. Uncomment the "installation" line in the first cell to install OpenSoundscape. |
| [![Download via DownGit](https://img.shields.io/badge/GitHub-Download-teal?logo=github)](https://minhaskamal.github.io/DownGit/#/home?url=https://github.com/kitzeslab/opensoundscape/blob/master/docs/tutorials/train_cnn.ipynb) | The link downloads the tutorial file to your computer. Follow the [Jupyter installation instructions](https://opensoundscape.org/en/latest/installation/jupyter.html), then open the tutorial file in Jupyter. |

In [1]:
# if this is a Google Colab notebook, install opensoundscape in the runtime environment
if 'google.colab' in str(get_ipython()):
  %pip install "opensoundscape==0.12.1" "jupyter-client<8,>=5.3.4" "ipykernel==6.17.1" "bioacoustics-model-zoo==0.12.0"
  num_workers=0
else:
  # choose cpu parallelization count
  num_workers=4

## Setup

### Import needed packages

In [2]:
#other utilities and packages
import torch
import pandas as pd
from pathlib import Path
import numpy as np
import pandas as pd
import random 
from glob import glob
import sklearn

from tqdm.autonotebook import tqdm
from sklearn.metrics import average_precision_score, roc_auc_score
from pathlib import Path

#set up plotting
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize']=[15,5] #for large visuals
%config InlineBackend.figure_format = 'retina'

# opensoundscape transfer learning tools
from opensoundscape.ml.shallow_classifier import MLPClassifier

/var/folders/d8/265wdp1n0bn_r85dh3pp95fh0000gq/T/ipykernel_49245/951004154.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


### Set random seeds

Set manual seeds for Pytorch and Python. These essentially "fix" the results of any stochastic steps in model training, ensuring that training results are reproducible. You probably don't want to do this when you actually train your model, but it's useful for debugging.

### Download and prepare training data


#### Download example files
Download a set of aquatic soundscape recordings with annotations of _Rana sierrae_ vocalizations

Option 1: run the cell below

- if you get a 403 error, DataDryad suspects you are a bot. Use Option 2. 

Option 2:

- Download and unzip the `rana_sierrae_2022.zip` folder containing audio and annotations from this [public Dryad dataset](https://datadryad.org/stash/dataset/doi:10.5061/dryad.9s4mw6mn3#readme)
- Move the unzipped `rana_sierrae_2022` folder into the current folder

In [3]:
# # Note: the "!" preceding each line below allows us to run bash commands in a Jupyter notebook
# # If you are not running this code in a notebook, input these commands into your terminal instead
# !wget -O rana_sierrae_2022.zip https://datadryad.org/stash/downloads/file_stream/2722802;
# !unzip rana_sierrae_2022;

#### Prepare audio data
See the train_cnn.ipynb tutorial for step-by-step walkthrough of this process, or just run the cells below to prepare a training set.

In [4]:
# Set this variable to specify where the folder `rana_sierrae_2022` is located:
dataset_path = Path("./rana_sierrae_2022/")

# let's generate clip labels of 5s duration (to match HawkEars) using the raven annotations
# and some utility functions from opensoundscape
from opensoundscape.annotations import BoxedAnnotations

audio_and_raven_files = pd.read_csv(f"{dataset_path}/audio_and_raven_files.csv")
# update the paths to where we have the audio and raven files stored
audio_and_raven_files["audio"] = audio_and_raven_files["audio"].apply(
    lambda x: f"{dataset_path}/{x}"
)
audio_and_raven_files["raven"] = audio_and_raven_files["raven"].apply(
    lambda x: f"{dataset_path}/{x}"
)

annotations = BoxedAnnotations.from_raven_files(
    raven_files=audio_and_raven_files["raven"],
    audio_files=audio_and_raven_files["audio"],
    annotation_column="annotation",
)
# generate labels for 5s clips, including any labels that overlap by at least 0.2 seconds
labels = annotations.clip_labels(clip_duration=5, min_label_overlap=0.2)

/Users/SML161/opensoundscape/opensoundscape/annotations.py:347: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_annotations_df = pd.concat(all_file_dfs).reset_index(drop=True)


#### Inspect labels

Count number of each annotation type: 

Note that the 'X' label is for when the annotator was uncertain about the identity of a call. Labels A-E denote distinct call types.

In [5]:
labels.sum()

A    512
E    128
D     62
B     24
C     74
X    118
dtype: int64

#### split into training and validation data
We'll just focus on class 'A', the call type with the most annotations. We'll randomly split the clips into training and validation data, acknowledging that this approach does not test the ability of the model to generalize. Since the samples in the training and validation sets could be adjascent 2-second audio clips, good performance could simply mean the model has memorized the training samples, and the validation set has very similar samples. 

In [6]:
labels_train, labels_val = sklearn.model_selection.train_test_split(labels[["A"]])

## Train shallow classifiers on embedding model outputs

We'll train our classifiers on a small annotated dataset with HawkEars Embedding Model, Perch, and BirdNET as feature extractors.

> Note: HawkEars Embedding model provides a single EfficientNet architecture, whereas the regular "HawkEars" class is an ensemble of 5 CNN architectures. For transfer learning tasks, the HawkEars_Embedding class is recommended.


In [7]:
import bioacoustics_model_zoo as bmz

hawk = bmz.HawkEars_Embedding()

/Users/SML161/opensoundscape/opensoundscape/preprocess/preprocessors.py:513: DeprecationWarning: sample_shape argument is deprecated. Please use height, width, channels arguments instead. 
                The current behavior is to override height, width, channels with sample_shape 
                when sample_shape is not None.
                
  warnings.warn(


Create a shallow classifier that we'll train with embeddings as inputs. The input size needs to match the size of the embeddings produced by our embedding model. HawkEars embeddings are vectors of length 2048. 

In [8]:
embedding_size = hawk.classifier.in_features

clf = MLPClassifier(
    input_size=embedding_size, output_size=labels_train.shape[1], hidden_layer_sizes=()
)

We can run a single function that will embed the training and validation samples, then train the classifier.

This will take a minute or two, since all of the samples need to be embedded with HawkEars. On a GPU it takes about 30 seconds to embed the samples.

In [9]:
emb_db, failed_samples = hawk.embed_to_hoplite_db(
    labels_train, "./rasi_hawkears_train_val", dataset_name="trainval", batch_size=128
)

Connecting to existing db at rasi_hawkears_train_val
Connected database has 1,344 embeddings from 1 dataset.
all samples already have embeddings in the database


In [10]:
# also embed validation set
emb_db, failed_samples = hawk.embed_to_hoplite_db(
    labels_val, "./rasi_hawkears_train_val", dataset_name="trainval", batch_size=128
)

Connecting to existing db at rasi_hawkears_train_val
Connected database has 1,344 embeddings from 1 dataset.
all samples already have embeddings in the database


train shallow classifier

In [11]:
from opensoundscape.ml.shallow_classifier import fit_on_hoplite_db

fit_on_hoplite_db(
    model=clf,
    hoplite_db=emb_db,
    dataset_name="trainval",
    train_df=labels_train,
    validation_df=labels_val,
    steps=100,
    batch_size=128,
)

Epoch 100/100, Loss: 0.349, Val Loss: 0.478
	val AU ROC: 0.822
	val MAP: 0.786
Loaded best model with validation loss: 0.477 at step 88 of 100
Training complete


{'loss': 0.47729257742563885,
 'auroc': 0.8223060177266365,
 'map': 0.785780395495743,
 'per_class_auroc': [0.8223060177266365]}